In [1]:
import pandas as pd 
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize
from scipy.sparse import csr_matrix

In [2]:
movies = pd.read_csv("../data/processed_movies.csv")
ratings = pd.read_csv("../data/ratings.csv")

movies['metadata'] = movies['metadata'].fillna('').astype(str)
display(movies.head(3))
display(ratings.head(3))

,movieId,title,genres,clean_title,tag,metadata
0,1,Toy Story (1995),Adventure Animation Children Comedy Fantasy,Toy Story,children Disney animation children Disney Disn...,Toy Story Adventure Animation Children Comedy...
1,2,Jumanji (1995),Adventure Children Fantasy,Jumanji,Robin Williams fantasy Robin Williams time tra...,Jumanji Adventure Children Fantasy Robin Will...
2,3,Grumpier Old Men (1995),Comedy Romance,Grumpier Old Men,comedinha de velhinhos engraÃƒÂ§ada comedinha ...,Grumpier Old Men Comedy Romance comedinha de ...


,userId,movieId,rating,timestamp
0,1,17,4.0,944249077
1,1,25,1.0,944250228
2,1,29,2.0,943230976


<H>USER BASED COLLABORATVIE FILTERING</H>

In [3]:
""""
user_movie_matrix = ratings.pivot_table(
    index='userId',
    columns='movieId',
    values='rating'
)
user_movie_filled = user_movie_matrix.fillna(0)
user_similarity = cosine_similarity(user_movie_filled)

user_similarity_df = pd.DataFrame(
    user_similarity,
    index=user_movie_matrix.index,
    columns=user_movie_matrix.index
)

display(user_movie_filled.head(3))
display(user_similarity_df.head(3))
"""

'"\nuser_movie_matrix = ratings.pivot_table(\n    index=\'userId\',\n    columns=\'movieId\',\n    values=\'rating\'\n)\nuser_movie_filled = user_movie_matrix.fillna(0)\nuser_similarity = cosine_similarity(user_movie_filled)\n\nuser_similarity_df = pd.DataFrame(\n    user_similarity,\n    index=user_movie_matrix.index,\n    columns=user_movie_matrix.index\n)\n\ndisplay(user_movie_filled.head(3))\ndisplay(user_similarity_df.head(3))\n'

In [4]:
movie_rating_counts = ratings.groupby('movieId').size()

movie_ids = np.sort(ratings['movieId'].unique())
user_ids = np.sort(ratings['userId'].unique())

movie_id_to_col = pd.Series(
    np.arange(len(movie_ids)),
    index=movie_ids
)

user_id_to_row = pd.Series(
    np.arange(len(user_ids)),
    index=user_ids
)

movie_col_to_id = pd.Series(
    movie_ids,
    index=np.arange(len(movie_ids))
)

user_row_to_id = pd.Series(
    user_ids,
    index=np.arange(len(user_ids))
)


In [5]:
user_rows = ratings['userId'].map(user_id_to_row).to_numpy()
movie_cols = ratings['movieId'].map(movie_id_to_col).to_numpy()
rating_values = ratings['rating'].astype('float32').to_numpy()

user_movie_sparse = csr_matrix(
    (
        rating_values,
        (user_rows, movie_cols)
    ),
    shape=(
        len(user_ids),
        len(movie_ids)
    ),
    dtype=np.float32
)

user_movie_norm = normalize(user_movie_sparse, axis=1)

In [6]:
print("User-movie sparse matrix shape:")
print(user_movie_sparse.shape)

print("Number of ratings:")
print(user_movie_sparse.nnz)

print("Users:", len(user_ids))
print("Movies:", len(movie_ids))

User-movie sparse matrix shape:
(200948, 84432)
Number of ratings:
32000204
Users: 200948
Movies: 84432


In [7]:
def get_similar_users(user_id, n=50):

    if user_id not in user_id_to_row.index:
        return "Invalid user ID"
    
    user_row = user_id_to_row[user_id]
    similarity_scores = (
        user_movie_norm[user_row]
        .dot(user_movie_norm.T)
        .toarray()
        .flatten()
    )
    similarity_scores[user_row] = -1
    sorted_indices = np.argsort(similarity_scores)[::-1]
    top_indices = sorted_indices[:n]
    similar_user_ids = user_row_to_id.iloc[top_indices].values
    similar_scores = similarity_scores[top_indices]

    similar_users = pd.Series(
        similar_scores,
        index=similar_user_ids
    )

    return similar_users

In [8]:
get_similar_users(5, n=10)

89910     0.926809
63473     0.926035
33975     0.924882
26814     0.909883
82162     0.907421
43123     0.907181
167865    0.906326
22577     0.905495
115408    0.904759
137490    0.903641
dtype: float32

In [9]:
#Basic collaborative filtering
def recommend_movies(user_id, n_movies=10, min_ratings=100, n_similar_users=200, round=4):

    if user_id not in user_id_to_row.index:
        return "Invalid user ID"
    
    similar_users = get_similar_users(user_id, n=n_similar_users)

    if isinstance(similar_users, str):
        return similar_users
    
    similar_users_ids = similar_users.index

    similar_users_ratings = ratings[ratings['userId'].isin(similar_users_ids)]
    mean_ratings = similar_users_ratings.groupby('movieId')['rating'].mean()

    watched_movies = ratings[ratings['userId'] == user_id]['movieId'].unique()
    recommendations = mean_ratings.drop(watched_movies, errors='ignore')

    recommendations = recommendations[
        movie_rating_counts.reindex(
            recommendations.index
        ).fillna(0) >= min_ratings
    ]

    recommendations = (
        recommendations
        .sort_values(ascending=False)
        .head(n_movies)
        
    )

    recommended_movies = movies[movies['movieId'].isin(recommendations.index)][['movieId', 'title', 'metadata']].copy()
    recommended_movies['predicted_rating'] = recommended_movies['movieId'].map(recommendations).round(round)
    recommended_movies = recommended_movies.sort_values(by='predicted_rating', ascending=False).reset_index(drop=True)

    return recommended_movies


In [10]:
#Weighted collaborative filtering
def weighted_recommend_movies(user_id, n_movies=10, min_ratings=100, n_similar_users=200, round=4):

     if user_id not in user_id_to_row.index:
        return "Invalid user ID"
     
     similar_users = get_similar_users(user_id, n=n_similar_users)

     if isinstance(similar_users, str):
        return similar_users
     
     similar_users_df = similar_users.reset_index()

     similar_users_df.columns = ['userId', 'similarity']

     similar_users_ratings = ratings[
         ratings['userId'].isin(similar_users.index)
         ].merge(
             similar_users_df,
             on='userId',
             how='left'
         )
     
     similar_users_ratings['weighted_ratings'] = similar_users_ratings['rating'] * similar_users_ratings['similarity']

     weighted_sum = (
         similar_users_ratings
         .groupby('movieId')['weighted_ratings']
         .sum()
     )
     similarity_sum = (
         similar_users_ratings
         .groupby('movieId')['similarity']
         .sum()
     )
     weighted_average = weighted_sum / similarity_sum

     watched_movies = ratings[ratings['userId'] == user_id]['movieId'].unique()

     recommendations = weighted_average.drop(watched_movies, errors='ignore')
     recommendations = recommendations[
         movie_rating_counts.reindex(recommendations.index).fillna(0) >= min_ratings
     ]
     recommendations = (
         recommendations
         .sort_values(ascending=False)
         .head(n_movies)
     )

     recommended_movies = movies[
        movies['movieId'].isin(recommendations.index)
     ][['movieId', 'title', 'metadata']].copy()

     recommended_movies['predicted_rating'] = (
         recommended_movies['movieId']
         .map(recommendations)
         .round(round)
     )
     
     recommended_movies = (
         recommended_movies
         .sort_values(by='predicted_rating', ascending=False)
         .reset_index(drop=True)
     )

     return recommended_movies

In [11]:
user_mean_ratings = ratings.groupby('userId')['rating'].mean()

display(user_mean_ratings.head(3))

userId
1    3.531915
2    4.269231
3    3.588435
Name: rating, dtype: float64

In [12]:
normalized_values = (
    ratings['rating'].astype('float32')
    -
    ratings['userId'].map(user_mean_ratings).astype('float32')
).to_numpy()

normalized_user_movie_sparse = csr_matrix(
    (
        normalized_values,
        (user_rows, movie_cols)
    ),
    shape=(len(user_ids), len(movie_ids)),
    dtype=np.float32
)

normalized_user_movie_norm = normalize(normalized_user_movie_sparse, axis=1)

In [13]:
def get_normalized_similar_users(user_id, n=50):

    if user_id not in user_id_to_row.index:
        return "Invalid user ID"

    user_row = user_id_to_row[user_id]

    similarity_scores = (
        normalized_user_movie_norm[user_row]
        .dot(normalized_user_movie_norm.T)
        .toarray()
        .flatten()
    )
    similarity_scores[user_row] = -1

    sorted_indices = np.argsort(similarity_scores)[::-1]
    top_indices = sorted_indices[:n]

    similar_user_ids = user_row_to_id.iloc[top_indices].values
    similar_scores = similarity_scores[top_indices]
    similar_users = pd.Series(similar_scores, index=similar_user_ids)

    return similar_users

In [14]:
def normalized_weighted_recommend_movies(user_id, n_movies=10, min_ratings=100, n_similar_users=200, round=4):

    if user_id not in user_id_to_row.index:
        return "Invalid user ID"
    
    similar_users = get_normalized_similar_users(user_id, n=n_similar_users)

    if isinstance(similar_users, str):
        return similar_users
    
    similar_users_df = similar_users.reset_index()
    similar_users_df.columns = ['userId', 'similarity']

    similar_users_ratings = ratings[
         ratings['userId'].isin(similar_users.index)
         ].merge(
             similar_users_df,
             on='userId',
             how='left'
         )
    
    similar_users_ratings['user_mean'] = similar_users_ratings['userId'].map(user_mean_ratings)
    similar_users_ratings['normalized_rating'] = similar_users_ratings['rating'] - similar_users_ratings['user_mean']
    similar_users_ratings['weighted_normalized_rating'] = similar_users_ratings['normalized_rating'] * similar_users_ratings['similarity']

    weighted_sum = (
        similar_users_ratings
        .groupby('movieId')['weighted_normalized_rating']
        .sum()
    )

    similarity_sum = (
        similar_users_ratings
        .assign(
            abs_similarity = lambda x: x['similarity'].abs()
        )
        .groupby('movieId')['abs_similarity']
        .sum()
    )

    predicted_scores = weighted_sum / similarity_sum
    predicted_scores = predicted_scores + user_mean_ratings[user_id]
    predicted_scores = (predicted_scores / predicted_scores.max()) * 5

    watched_movies = ratings[ratings['userId'] == user_id]['movieId'].unique()
    recommendations = predicted_scores.drop(watched_movies, errors='ignore')

    recommendations = recommendations[movie_rating_counts.reindex(recommendations.index).fillna(0) >= min_ratings]
    recommendations = recommendations.sort_values(ascending=False).head(n_movies)

    recommended_movies = movies[
        movies['movieId'].isin(recommendations.index)
    ][['movieId', 'title', 'metadata']].copy()

    recommended_movies['predicted_rating'] = (
        recommended_movies['movieId']
        .map(recommendations)
        .round(round)
    )

    recommended_movies = (
        recommended_movies
        .sort_values(by='predicted_rating', ascending=False)
        .reset_index(drop=True)
    )

    return recommended_movies

In [15]:
recommend_movies(1, n_movies=10, min_ratings=100, n_similar_users=500, round=3)

,movieId,title,metadata,predicted_rating
0,1153,Raw Deal (1948),Raw Deal Film-Noir film noir tcm tivo18 film ...,5.0
1,1444,Guantanamera (1994),Guantanamera Comedy,5.0
2,1549,Rough Magic (1995),Rough Magic Drama Romance Russell Crowe mexic...,5.0
3,1780,Ayn Rand: A Sense of Life (1997),Ayn Rand: A Sense of Life Documentary biograp...,5.0
4,7013,"Night of the Hunter, The (1955)","Night of the Hunter, The Drama Film-Noir Thri...",5.0
5,7080,42nd Street (1933),42nd Street Drama Musical Romance Tumey's DVD...,5.0
6,7121,Adam's Rib (1949),Adam's Rib Comedy Romance Ahead of its time c...,5.0
7,7444,13 Going on 30 (2004),13 Going on 30 Comedy Fantasy Romance best fr...,5.0
8,7619,"Miracle Worker, The (1962)","Miracle Worker, The Drama Andrew Prine Anne B...",5.0
9,8125,Sunrise: A Song of Two Humans (1927),Sunrise: A Song of Two Humans Drama Romance a...,5.0


In [16]:
weighted_recommend_movies(1, n_movies=10, min_ratings=100, n_similar_users=500, round=3)

,movieId,title,metadata,predicted_rating
0,834,Phat Beach (1996),Phat Beach Comedy african american beach merc...,5.0
1,1153,Raw Deal (1948),Raw Deal Film-Noir film noir tcm tivo18 film ...,5.0
2,1444,Guantanamera (1994),Guantanamera Comedy,5.0
3,1549,Rough Magic (1995),Rough Magic Drama Romance Russell Crowe mexic...,5.0
4,3338,For All Mankind (1989),For All Mankind Documentary Sundance Audience...,5.0
5,7013,"Night of the Hunter, The (1955)","Night of the Hunter, The Drama Film-Noir Thri...",5.0
6,7080,42nd Street (1933),42nd Street Drama Musical Romance Tumey's DVD...,5.0
7,7444,13 Going on 30 (2004),13 Going on 30 Comedy Fantasy Romance best fr...,5.0
8,7619,"Miracle Worker, The (1962)","Miracle Worker, The Drama Andrew Prine Anne B...",5.0
9,8125,Sunrise: A Song of Two Humans (1927),Sunrise: A Song of Two Humans Drama Romance a...,5.0


In [17]:
normalized_weighted_recommend_movies(1, n_movies=10, min_ratings=100, n_similar_users=500, round=3)

,movieId,title,metadata,predicted_rating
0,751,Careful (1992),Careful Comedy Horror incest not funny not sc...,5.000
1,790,"Unforgettable Summer, An (Un été inoubliable) ...","Unforgettable Summer, An (Un été inoubliable) ...",5.000
2,2557,I Stand Alone (Seul contre tous) (1998),I Stand Alone (Seul contre tous) Drama Thrill...,5.000
3,4393,Another Woman (1988),Another Woman Drama cerebral characters famil...,4.990
4,44694,Volver (2006),Volver Comedy Drama Tumey's DVDs eomwn life i...,4.903
5,95558,Beasts of the Southern Wild (2012),Beasts of the Southern Wild Drama Fantasy emo...,4.903
6,6791,Babette's Feast (Babettes gæstebud) (1987),Babette's Feast (Babettes gæstebud) Drama Cri...,4.810
7,1570,Tetsuo II: Body Hammer (1992),Tetsuo II: Body Hammer Horror Sci-Fi body hor...,4.787
8,3823,Wonderland (1999),Wonderland Drama easily confused with other m...,4.765
9,4298,Rififi (Du rififi chez les hommes) (1955),Rififi (Du rififi chez les hommes) Crime Film...,4.721


<H>ITEM BASED COLLABORATIVE FILTERING</H>

In [18]:
movie_user_sparse = user_movie_sparse.T.tocsr()
movie_user_norm = normalize(movie_user_sparse, axis=1)

In [19]:
def find_movie(title):
    
    matches = movies[
        movies['title']
        .str.lower()
        .str.contains(title.lower(), regex=False)
    ]

    if matches.empty:
        return "Movie not found."
    
    return matches.iloc[0]

In [20]:
def recommend_similar_movies(title, n_movies=10, min_ratings=100, round=4):
    
    movie = find_movie(title)

    if movie is None:
        return "Movie not found."
    
    title = movie['title']
    movie_id = movie['movieId']

    print(f"Recommending for: {title}")

    if movie_id not in movie_id_to_col.index:
        return "This movie has no rating data"
    
    movie_col = movie_id_to_col[movie_id]

    similarity_scores = (
        movie_user_norm[movie_col]
        .dot(movie_user_norm.T)
        .toarray()
        .flatten()
    )
    similarity_scores[movie_col] = -1

    sorted_indices = np.argsort(similarity_scores)[::-1]
    candidate_movie_ids = movie_col_to_id.iloc[sorted_indices].values
    candidate_scores = similarity_scores[sorted_indices]

    similar_movies = pd.Series(candidate_scores, index=candidate_movie_ids)
    similar_movies = similar_movies[similar_movies > 0]
    similar_movies = similar_movies[
        movie_rating_counts.reindex(similar_movies.index).fillna(0) >= min_ratings
    ]
    similar_movies = similar_movies.head(n_movies)

    recommended_movies = movies[
        movies['movieId'].isin(similar_movies.index)
    ][['movieId', 'title', 'metadata']].copy()

    recommended_movies['similarity_score'] = (
        recommended_movies['movieId']
        .map(similar_movies)
        .round(round)
    )

    recommended_movies = (
        recommended_movies
        .sort_values(by='similarity_score', ascending=False)
        .reset_index(drop=True)
    )
    
    return recommended_movies


In [ ]:
recommend_similar_movies('Toy Story', n_movies=10, min_ratings=100, round=3)

Recommending for: Toy Story (1995)


,movieId,title,metadata,similarity_score
0,3114,Toy Story 2 (1999),Toy Story 2 Adventure Animation Children Come...,0.575
1,260,Star Wars: Episode IV - A New Hope (1977),Star Wars: Episode IV - A New Hope Action Adv...,0.562
2,1270,Back to the Future (1985),Back to the Future Adventure Comedy Sci-Fi 19...,0.545
3,356,Forrest Gump (1994),Forrest Gump Comedy Drama Romance War bitters...,0.542
4,480,Jurassic Park (1993),Jurassic Park Action Adventure Sci-Fi Thrille...,0.540
5,364,"Lion King, The (1994)","Lion King, The Adventure Animation Children D...",0.537
6,1210,Star Wars: Episode VI - Return of the Jedi (1983),Star Wars: Episode VI - Return of the Jedi Ac...,0.534
7,780,Independence Day (a.k.a. ID4) (1996),Independence Day (a.k.a. ID4) Action Adventur...,0.531
8,588,Aladdin (1992),Aladdin Adventure Animation Children Comedy M...,0.527
9,4306,Shrek (2001),Shrek Adventure Animation Children Comedy Fan...,0.514


In [ ]:
recommend_similar_movies('Shrek', n_movies=10, min_ratings=100, round=3)

Recommending for: Shrek (2001)


,movieId,title,metadata,similarity_score
0,4886,"Monsters, Inc. (2001)","Monsters, Inc. Adventure Animation Children C...",0.679
1,6377,Finding Nemo (2003),Finding Nemo Adventure Animation Children Com...,0.664
2,6539,Pirates of the Caribbean: The Curse of the Bla...,Pirates of the Caribbean: The Curse of the Bla...,0.629
3,4993,"Lord of the Rings: The Fellowship of the Ring,...","Lord of the Rings: The Fellowship of the Ring,...",0.625
4,8360,Shrek 2 (2004),Shrek 2 Adventure Animation Children Comedy M...,0.623
5,5952,"Lord of the Rings: The Two Towers, The (2002)","Lord of the Rings: The Two Towers, The Advent...",0.604
6,8961,"Incredibles, The (2004)","Incredibles, The Action Adventure Animation C...",0.603
7,5349,Spider-Man (2002),Spider-Man Action Adventure Sci-Fi Thriller d...,0.592
8,7153,"Lord of the Rings: The Return of the King, The...","Lord of the Rings: The Return of the King, The...",0.580
9,4963,Ocean's Eleven (2001),Ocean's Eleven Crime Thriller Brad Pitt con m...,0.568
